# **K-Means vs K-Means++ Initialisation in kmeans_tpu**

This notebook compares random and K-Means++ initialisation strategies
provided by `kmeans_tpu`, showing the effect on clustering quality
and visual results.

> **Note:** This notebook requires a TPU runtime.
> In Colab: Runtime → Change runtime type → TPU v5e-1 TPU

> **Compatibility:** This notebook has been tested with JAX 0.7.2 and
> Python 3.13 on Google Colab.

## 1. Installation and setup

In [ ]:
!pip install git+https://github.com/vbrachetta/kmeans_tpu.git

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from kmeans_tpu import kmeans, inertia

print(f"JAX version: {jax.__version__}")
print(f"Available devices: {jax.devices()}")

## 2. Generate well-separated clusters

We use `make_blobs` to generate a 2D dataset with 3 well-separated clusters,
which allows us to visualise the results directly.

In [ ]:
data_np, true_labels = make_blobs(
    n_samples=300,
    n_features=2,
    centers=3,
    random_state=42,
)
data = jnp.array(data_np.astype("float32"))

print(f"Data shape : {data.shape}")
print(f"Data dtype : {data.dtype}")

## 3. Visualise the data before clustering

Before running any algorithm, let us look at the raw data with true labels.

In [ ]:
plt.figure(figsize=(5, 4))
plt.scatter(data_np[:, 0], data_np[:, 1], c=true_labels, s=20, alpha=0.7)
plt.title("Raw Data — True Labels")
plt.tight_layout()
plt.show()

## 4. Run both initialisation strategies

We run K-Means with both random and K-Means++ initialisation on the same
dataset and compare the results.

In [ ]:
centroids_rnd, assignments_rnd, n_iter_rnd = kmeans(data, k=3, init="random")
centroids_pp,  assignments_pp,  n_iter_pp  = kmeans(data, k=3, init="plusplus")

score_rnd = inertia(data, centroids_rnd, assignments_rnd)
score_pp  = inertia(data, centroids_pp,  assignments_pp)

print(f"Random    — Inertia: {score_rnd:.2f}, Iterations: {n_iter_rnd}")
print(f"K-Means++ — Inertia: {score_pp:.2f},  Iterations: {n_iter_pp}")

## 5. Visual comparison

Let us visualise the clustering results side by side, alongside the true labels.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Random initialisation
axes[0].scatter(data_np[:, 0], data_np[:, 1],
                c=np.array(assignments_rnd), s=20, alpha=0.7)
axes[0].scatter(np.array(centroids_rnd)[:, 0],
                np.array(centroids_rnd)[:, 1],
                c="black", s=200, marker="X", zorder=5, label="Centroids")
axes[0].set_title(f"Random (inertia={score_rnd:.0f}, iter={n_iter_rnd})")
axes[0].legend()

# K-Means++ initialisation
axes[1].scatter(data_np[:, 0], data_np[:, 1],
                c=np.array(assignments_pp), s=20, alpha=0.7)
axes[1].scatter(np.array(centroids_pp)[:, 0],
                np.array(centroids_pp)[:, 1],
                c="black", s=200, marker="X", zorder=5, label="Centroids")
axes[1].set_title(f"K-Means++ (inertia={score_pp:.0f}, iter={n_iter_pp})")
axes[1].legend()

plt.suptitle("K-Means vs K-Means++ — Clustering Results", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Summary

K-Means++ typically achieves lower inertia and converges in fewer iterations
than random initialisation, at the cost of a slightly more expensive initialisation step. On TPU hardware this difference is negligible in practice.

For further details on the initialisation strategies, see the
[README](https://github.com/vbrachetta/kmeans_tpu).